# Full Component Labeller — Shaft / Crossarm / Transformer / Primary / Secondary / Neutral / Downguy / Anchor

Reads `allwires_v1.las`, generates a **LiDAR-native 2D side-elevation render per pole**
showing ALL nearby points, and lets you paint all 8 component labels.

### Label keys
| Key | Component | LAS out | Colour |
|---|---|---|---|
| `1` | SHAFT | 65 | Red |
| `2` | CROSSARM | 66 | Cyan |
| `3` | PRIMARY wire | 67 | Bright green |
| `4` | SECONDARY wire | 68 | Dark green |
| `5` | NEUTRAL wire | 69 | Yellow |
| `6` | DOWNGUY | 72 | Magenta |
| `7` | ANCHOR | 73 | Orange |
| `8` | TRANSFORMER | 74 | Blue |

### Brush controls
| Key | Action |
|---|---|
| `[` / `]` | Smaller / larger brush |
| Right-click | Undo last stroke |
| `S` | Save mask → next pole |
| `Q` | Quit (saves progress so far) |

### Workflow
1. **Cell 1** — load LAS, build pole index
2. **Cell 2** — generate all pole renders → `pole_renders/`
3. **Cell 3** — interactive labeler (run in local Jupyter with display)
4. **Cell 4** — back-project masks → 3D → save `allwires_labelled_full.las`
5. **Cell 5** — sanity check


## Cell 1 — Load LAS + build pole index

In [1]:
import os, cv2
import numpy as np
import laspy
from sklearn.decomposition import PCA
from scipy.spatial import cKDTree

IN_LAS     = 'allwires_v1.las'
OUT_LAS    = 'allwires_labelled_full.las'
OUT_QML    = 'allwires_labelled_full.qml'
RENDER_DIR = 'pole_renders'
MASK_DIR   = 'pole_masks'
os.makedirs(RENDER_DIR, exist_ok=True)
os.makedirs(MASK_DIR,   exist_ok=True)

IMG_W = 640   # render width  (pixels — horizontal = road direction)
IMG_H = 896   # render height (pixels — vertical   = Z)

# ── 8 label classes — key → (name, BGR colour, output LAS class) ──────────────
LABEL_CLASSES = {
    1: ('SHAFT',       (60,  60,  220), 65),   # red
    2: ('CROSSARM',    (220, 220, 0  ), 66),   # cyan
    3: ('PRIMARY',     (0,   255, 0  ), 67),   # bright green
    4: ('SECONDARY',   (0,   180, 0  ), 68),   # dark green
    5: ('NEUTRAL',     (0,   255, 255), 69),   # yellow
    6: ('DOWNGUY',     (255, 0,   255), 72),   # magenta
    7: ('ANCHOR',      (0,   140, 255), 73),   # orange
    8: ('TRANSFORMER', (255, 0,   0  ), 74),   # blue
}

# ── Background colours for existing LAS classes shown in render ───────────────
# These are dim so the user can distinguish them from painted labels
BG_COLORS = {
    65: (40,  140, 40 ),   # dim green  — existing POLE (shaft+crossarm+transformer mixed)
    70: (160, 90,  40 ),   # dim blue   — SAME_LANE_WIRE
    71: (40,  40,  160),   # dim red    — CROSS_ROAD_WIRE
    72: (160, 0,   160),   # dim magenta — DOWNGUY
}

# ── Load LAS ──────────────────────────────────────────────────────────────────
print(f'Loading {IN_LAS} ...')
las = laspy.read(IN_LAS)
pts = np.vstack([las.x, las.y, las.z]).T        # (N, 3)
cls = np.array(las.classification, dtype=np.uint8)
uid = np.array(las.user_data,      dtype=np.int32)
print(f'  {len(pts):,} points')
for c, cnt in zip(*np.unique(cls, return_counts=True)):
    names = {65:'POLE',70:'SAME_LANE',71:'CROSS_ROAD',72:'DOWNGUY'}
    print(f'  Class {c:3d}  {names.get(int(c),"?"):>12s} : {cnt:>7,}')

# ── Per-pole info: centroid + Z range from uid field ─────────────────────────
pole_mask = cls == 65
pole_ids  = np.unique(uid[pole_mask])
pole_ids  = pole_ids[pole_ids > 0]

pole_info = {}
for pid in pole_ids:
    m = pole_mask & (uid == pid)
    p = pts[m]
    pole_info[pid] = {
        'xy'   : p[:, :2].mean(axis=0),
        'top_z': float(p[:, 2].max()),
        'bot_z': float(p[:, 2].min()),
    }

# ── Full-cloud KDTree for render candidate pull ───────────────────────────────
full_tree = cKDTree(pts[:, :2])

print(f'\n{len(pole_info)} poles indexed. Run Cell 2 to generate renders.')


Loading allwires_v1.las ...
  98,262 points
  Class  65          POLE :  83,754
  Class  70     SAME_LANE :   8,527
  Class  71    CROSS_ROAD :   4,768
  Class  72       DOWNGUY :   1,213

37 poles indexed. Run Cell 2 to generate renders.


## Cell 2 — Generate pole renders (all components visible)

In [2]:
# ── Render params ─────────────────────────────────────────────────────────────
RENDER_RADIUS = 60.0   # m — XY radius to pull ALL points for the render
                       # Wide: captures wires running along the road + anchor pts
Z_MARGIN      = 2.0    # m — extra space above/below pole in render
PT_R          = 2      # pixel dot radius

def make_render(pid):
    """
    LiDAR-native 2D side-elevation render.
    - Pulls ALL raw points within RENDER_RADIUS of the pole centre
    - Horizontal axis = 1st PCA component of all XY coords (= road direction)
    - Vertical axis   = Z, flipped (top of image = highest point)
    - Points drawn in dim BG_COLORS by existing class
    - Returns: img_bgr, pca, u_min, u_max, z_lo, z_hi, nearby_global_idx
    """
    info    = pole_info[pid]
    pole_xy = info['xy']
    top_z   = info['top_z']
    bot_z   = info['bot_z']
    z_lo    = bot_z - Z_MARGIN
    z_hi    = top_z + Z_MARGIN

    # Pull all points within RENDER_RADIUS
    raw_local = np.array(full_tree.query_ball_point(pole_xy, r=RENDER_RADIUS), dtype=np.int64)
    if len(raw_local) == 0:
        return None

    near_pts = pts[raw_local]
    near_cls = cls[raw_local]

    # Z window filter
    zm = (near_pts[:, 2] >= z_lo) & (near_pts[:, 2] <= z_hi)
    near_pts  = near_pts[zm]
    near_cls  = near_cls[zm]
    near_gidx = raw_local[zm]

    if len(near_pts) == 0:
        return None

    # PCA on XY → road direction as horizontal axis
    pca   = PCA(n_components=2).fit(near_pts[:, :2])
    u     = pca.transform(near_pts[:, :2])[:, 0]
    z     = near_pts[:, 2]

    u_min, u_max = u.min(), u.max()
    u_range = max(u_max - u_min, 1e-3)
    z_range = max(z_hi - z_lo, 1e-3)

    def to_px(u_arr, z_arr):
        ux = np.clip(((u_arr - u_min) / u_range * (IMG_W - 1)).astype(int), 0, IMG_W - 1)
        vx = np.clip(((1.0 - (z_arr - z_lo) / z_range) * (IMG_H - 1)).astype(int), 0, IMG_H - 1)
        return ux, vx

    # ── Draw background (dim existing classes, draw order: wires first, pole on top)
    img = np.full((IMG_H, IMG_W, 3), 18, dtype=np.uint8)   # dark background

    draw_order = [70, 71, 72, 65]   # wires → downguy → pole (pole on top)
    for draw_cls in draw_order:
        m = near_cls == draw_cls
        if not m.any():
            continue
        color = BG_COLORS.get(draw_cls, (80, 80, 80))
        ux, vx = to_px(u[m], z[m])
        for x, y in zip(ux, vx):
            cv2.circle(img, (int(x), int(y)), PT_R, color, -1)

    # ── Pole centre vertical line (grey guide) ────────────────────────────────
    pole_u_px = int((pca.transform(pole_xy.reshape(1,-1))[0,0] - u_min) / u_range * (IMG_W-1))
    pole_u_px = np.clip(pole_u_px, 0, IMG_W-1)
    cv2.line(img, (pole_u_px, 0), (pole_u_px, IMG_H-1), (50, 50, 50), 1)

    # ── Horizontal Z guide lines every 1m ────────────────────────────────────
    for dz in np.arange(0, z_range, 1.0):
        vy = int((1.0 - dz / z_range) * (IMG_H - 1))
        cv2.line(img, (0, vy), (IMG_W-1, vy), (30, 30, 30), 1)

    # ── Z labels on right edge ────────────────────────────────────────────────
    for dz in np.arange(0, z_range, 2.0):
        vy   = int((1.0 - dz / z_range) * (IMG_H - 1))
        z_lbl= z_lo + dz
        cv2.putText(img, f'{z_lbl:.0f}m', (IMG_W - 42, max(vy-2, 10)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.3, (100, 100, 100), 1)

    # ── Legend (top-left) ─────────────────────────────────────────────────────
    for k, (key, (name, color, _)) in enumerate(LABEL_CLASSES.items()):
        cv2.rectangle(img, (4, 4 + k*17), (13, 13 + k*17), color, -1)
        cv2.putText(img, f'{key}={name}', (16, 13 + k*17),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.32, (200, 200, 200), 1)

    # ── Pole ID header ────────────────────────────────────────────────────────
    cv2.putText(img, f'Pole {pid}  (top={top_z:.1f}m  h={top_z-bot_z:.1f}m)',
                (IMG_W//2 - 80, 16),
                cv2.FONT_HERSHEY_SIMPLEX, 0.42, (255, 255, 255), 1)

    return img, pca, u_min, u_max, z_lo, z_hi, near_gidx

# ── Generate all renders ──────────────────────────────────────────────────────
render_meta = {}

print(f'Generating {len(pole_info)} renders → {RENDER_DIR}/')
for pid in sorted(pole_info.keys()):
    result = make_render(pid)
    if result is None:
        print(f'  Pole {pid}: no points found — skipped')
        continue
    img, pca, u_min, u_max, z_lo, z_hi, near_gidx = result
    out_path = os.path.join(RENDER_DIR, f'pole_{pid:03d}.png')
    cv2.imwrite(out_path, img)
    render_meta[pid] = (pca, u_min, u_max, z_lo, z_hi, near_gidx)
    print(f'  Pole {pid:3d}: {len(near_gidx):,} pts → {os.path.basename(out_path)}')

print(f'\nDone. Check {RENDER_DIR}/ — then run Cell 3.')


Generating 37 renders → pole_renders/
  Pole   1: 3,975 pts → pole_001.png
  Pole   2: 4,901 pts → pole_002.png
  Pole   4: 4,622 pts → pole_004.png
  Pole   5: 8,899 pts → pole_005.png
  Pole   6: 8,778 pts → pole_006.png
  Pole   7: 8,322 pts → pole_007.png


  Pole  10: 10,509 pts → pole_010.png


  Pole  11: 10,279 pts → pole_011.png
  Pole  12: 5,939 pts → pole_012.png
  Pole  13: 3,634 pts → pole_013.png
  Pole  14: 3,412 pts → pole_014.png
  Pole  15: 2,245 pts → pole_015.png
  Pole  16: 5,292 pts → pole_016.png
  Pole  17: 9,149 pts → pole_017.png
  Pole  18: 8,803 pts → pole_018.png
  Pole  20: 14,711 pts → pole_020.png


  Pole  23: 13,434 pts → pole_023.png
  Pole  24: 13,064 pts → pole_024.png
  Pole  25: 9,865 pts → pole_025.png
  Pole  26: 20,904 pts → pole_026.png
  Pole  28: 19,832 pts → pole_028.png
  Pole  29: 18,496 pts → pole_029.png


  Pole  30: 29,985 pts → pole_030.png
  Pole  31: 22,242 pts → pole_031.png
  Pole  32: 22,642 pts → pole_032.png
  Pole  33: 20,746 pts → pole_033.png
  Pole  34: 9,517 pts → pole_034.png


  Pole  37: 13,609 pts → pole_037.png
  Pole  38: 17,368 pts → pole_038.png
  Pole  39: 6,767 pts → pole_039.png
  Pole  41: 12,944 pts → pole_041.png
  Pole  42: 15,293 pts → pole_042.png
  Pole  43: 16,168 pts → pole_043.png
  Pole  44: 18,746 pts → pole_044.png


  Pole  45: 13,765 pts → pole_045.png
  Pole  46: 13,347 pts → pole_046.png
  Pole  47: 12,046 pts → pole_047.png

Done. Check pole_renders/ — then run Cell 3.


## Cell 3 — Interactive labeler

In [3]:
# ── Full 8-class semantic labeler ────────────────────────────────────────────
def semantic_labeler(img_path, mask_path):
    """Returns True=saved/continue, False=quit."""
    img_orig = cv2.imread(img_path)
    if img_orig is None:
        print(f'  Cannot read {img_path} — skipping'); return True

    img  = img_orig.copy()
    mask = np.zeros(img.shape[:2], dtype=np.uint8)
    img_hist  = [img.copy()]
    mask_hist = [mask.copy()]
    active_id = 1
    brush     = 8
    drawing   = False

    def paint(event, x, y, flags, _):
        nonlocal drawing, brush
        if event == cv2.EVENT_LBUTTONDOWN:
            img_hist.append(img.copy()); mask_hist.append(mask.copy())
            drawing = True
            if len(img_hist) > 40: img_hist.pop(0); mask_hist.pop(0)
        elif event == cv2.EVENT_LBUTTONUP:
            drawing = False
        elif event == cv2.EVENT_MOUSEMOVE and drawing:
            name, color, _ = LABEL_CLASSES[active_id]
            cv2.circle(mask, (x, y), brush, active_id, -1)
            cv2.circle(img,  (x, y), brush, color,     -1)
        elif event == cv2.EVENT_RBUTTONDOWN:
            if len(img_hist) > 1:
                img[:]  = img_hist.pop()
                mask[:] = mask_hist.pop()

    win = 'Labeler | 1-8:class  []:brush  RClick:undo  S:save  Q:quit'
    cv2.namedWindow(win, cv2.WINDOW_NORMAL)
    cv2.resizeWindow(win, IMG_W, IMG_H)
    cv2.setMouseCallback(win, paint)

    while True:
        disp = img.copy()
        name, color, las_id = LABEL_CLASSES[active_id]
        # Status bar at bottom
        cv2.rectangle(disp, (0, IMG_H-22), (IMG_W, IMG_H), (30,30,30), -1)
        cv2.circle(disp, (10, IMG_H-11), 8, color, -1)
        cv2.putText(disp, f'[{active_id}] {name}  (LAS={las_id})   brush={brush}px',
                    (24, IMG_H-6), cv2.FONT_HERSHEY_SIMPLEX, 0.42, (220,220,220), 1)
        cv2.imshow(win, disp)

        k = cv2.waitKey(1) & 0xFF
        if ord('1') <= k <= ord('8'):
            active_id = int(chr(k))
        elif k == ord('['):
            brush = max(1, brush - 1)
        elif k == ord(']'):
            brush = min(80, brush + 1)
        elif k == ord('s'):
            cv2.imwrite(mask_path, mask)
            cv2.destroyWindow(win)
            return True
        elif k == ord('q'):
            cv2.destroyAllWindows()
            return False

    cv2.destroyAllWindows()
    return True

# ── Run labeler for each pole ─────────────────────────────────────────────────
labelled_pids = []
print(f'Starting labeller — {len(render_meta)} poles.')
print('Paint every visible component. Press S when done with a pole.\n')

for pid in sorted(render_meta.keys()):
    img_path  = os.path.join(RENDER_DIR, f'pole_{pid:03d}.png')
    mask_path = os.path.join(MASK_DIR,   f'pole_{pid:03d}_mask.png')

    if os.path.exists(mask_path):
        print(f'  Pole {pid:3d}: already masked — skipping (delete file to redo)')
        labelled_pids.append(pid)
        continue

    print(f'  Opening pole {pid} ...')
    saved = semantic_labeler(img_path, mask_path)
    if saved:
        labelled_pids.append(pid)
    else:
        print(f'  Quit at pole {pid}. Run Cell 4 to save progress.')
        break

print(f'\n{len(labelled_pids)}/{len(render_meta)} poles labelled. Run Cell 4.')


Starting labeller — 37 poles.
Paint every visible component. Press S when done with a pole.

  Pole   1: already masked — skipping (delete file to redo)
  Pole   2: already masked — skipping (delete file to redo)
  Pole   4: already masked — skipping (delete file to redo)
  Pole   5: already masked — skipping (delete file to redo)
  Pole   6: already masked — skipping (delete file to redo)
  Pole   7: already masked — skipping (delete file to redo)
  Pole  10: already masked — skipping (delete file to redo)
  Pole  11: already masked — skipping (delete file to redo)
  Pole  12: already masked — skipping (delete file to redo)
  Pole  13: already masked — skipping (delete file to redo)
  Pole  14: already masked — skipping (delete file to redo)
  Pole  15: already masked — skipping (delete file to redo)
  Pole  16: already masked — skipping (delete file to redo)
  Pole  17: already masked — skipping (delete file to redo)
  Pole  18: already masked — skipping (delete file to redo)
  Pole 

  Quit at pole 38. Run Cell 4 to save progress.

28/37 poles labelled. Run Cell 4.


## Cell 4 — Back-project masks → 3D → save LAS

In [4]:
# ── Build label → LAS class lookup ──────────────────────────────────────────
LABEL_TO_LAS = {k: v[2] for k, v in LABEL_CLASSES.items()}   # {1:65, 2:66, ...}

# Start from original classifications — unlabelled points keep their class
new_cls = cls.copy()
total_relabelled = 0

for pid in sorted(render_meta.keys()):
    mask_path = os.path.join(MASK_DIR, f'pole_{pid:03d}_mask.png')
    if not os.path.exists(mask_path):
        continue

    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    if mask is None or mask.max() == 0:
        continue   # empty mask

    pca, u_min, u_max, z_lo, z_hi, near_gidx = render_meta[pid]

    # Project all near points using IDENTICAL transform to Cell 2
    near_pts_3d = pts[near_gidx]
    u     = pca.transform(near_pts_3d[:, :2])[:, 0]
    z     = near_pts_3d[:, 2]
    u_range = max(u_max - u_min, 1e-3)
    z_range = max(z_hi - z_lo, 1e-3)

    ux = np.clip(((u - u_min) / u_range * (IMG_W - 1)).astype(int), 0, IMG_W - 1)
    vx = np.clip(((1.0 - (z - z_lo) / z_range) * (IMG_H - 1)).astype(int), 0, IMG_H - 1)

    painted = mask[vx, ux]   # label id at each point's pixel (0 = not painted)

    n = 0
    for local_i, label_id in enumerate(painted):
        if label_id == 0 or label_id not in LABEL_TO_LAS:
            continue
        new_cls[near_gidx[local_i]] = LABEL_TO_LAS[label_id]
        n += 1
    total_relabelled += n
    print(f'  Pole {pid:3d}: {n:,} pts relabelled')

print(f'\nTotal relabelled: {total_relabelled:,}')
for c, cnt in zip(*np.unique(new_cls, return_counts=True)):
    names = {65:'SHAFT',66:'CROSSARM',67:'PRIMARY',68:'SECONDARY',69:'NEUTRAL',
             70:'SAME_LANE',71:'CROSS_ROAD',72:'DOWNGUY',73:'ANCHOR',74:'TRANSFORMER'}
    print(f'  Class {c:3d}  {names.get(int(c),"?"):>12s} : {cnt:>7,}')

# ── Write LAS ─────────────────────────────────────────────────────────────────
header        = laspy.LasHeader(point_format=las.header.point_format, version=las.header.version)
header.scales = las.header.scales; header.offsets = las.header.offsets
out           = laspy.LasData(header)
out.x, out.y, out.z = pts.T
out.classification   = new_cls
out.user_data        = uid
out.write(OUT_LAS)
print(f'\nSaved → {OUT_LAS}')

# ── Write QML ─────────────────────────────────────────────────────────────────
entries = [
    (65, '60,220,60,255',   'SHAFT'),
    (66, '220,220,0,255',   'CROSSARM'),
    (67, '0,255,0,255',     'PRIMARY'),
    (68, '0,180,0,255',     'SECONDARY'),
    (69, '0,255,255,255',   'NEUTRAL'),
    (70, '60,130,220,255',  'SAME_LANE_WIRE'),
    (71, '220,60,60,255',   'CROSS_ROAD_WIRE'),
    (72, '255,0,255,255',   'DOWNGUY'),
    (73, '0,140,255,255',   'ANCHOR'),
    (74, '255,0,0,255',     'TRANSFORMER'),
]
cats = '\n'.join(f'      <category value="{v}" label="{n}" color="{c}" render="true"/>'
                  for v,c,n in entries)
qml = f"""<!DOCTYPE qgis PUBLIC 'http://mrcc.com/qgis.dtd' 'SYSTEM'>
<qgis styleCategories="Symbology" version="3.34">
  <renderer-v2 type="pointcloudclassified" attribute="Classification">
    <categories>
{cats}
    </categories>
  </renderer-v2>
</qgis>
"""
with open(OUT_QML, 'w') as f: f.write(qml)
print(f'Saved → {OUT_QML}')
print('\n✅  QGIS → Layer → Load Style → allwires_labelled_full.qml')


  Pole   1: 384 pts relabelled
  Pole   2: 55 pts relabelled
  Pole   4: 32 pts relabelled
  Pole   5: 0 pts relabelled
  Pole   6: 0 pts relabelled
  Pole   7: 136 pts relabelled
  Pole  10: 283 pts relabelled


  Pole  11: 1,077 pts relabelled
  Pole  12: 533 pts relabelled
  Pole  13: 216 pts relabelled
  Pole  14: 51 pts relabelled
  Pole  15: 31 pts relabelled
  Pole  16: 5 pts relabelled
  Pole  17: 136 pts relabelled


  Pole  18: 87 pts relabelled
  Pole  20: 6,181 pts relabelled
  Pole  23: 285 pts relabelled
  Pole  24: 100 pts relabelled
  Pole  25: 0 pts relabelled
  Pole  26: 0 pts relabelled


  Pole  28: 434 pts relabelled
  Pole  29: 29 pts relabelled
  Pole  30: 5,487 pts relabelled
  Pole  31: 249 pts relabelled
  Pole  32: 242 pts relabelled
  Pole  33: 0 pts relabelled
  Pole  34: 201 pts relabelled
  Pole  37: 0 pts relabelled

Total relabelled: 16,234
  Class  65         SHAFT :  84,012
  Class  66      CROSSARM :   1,026
  Class  68     SECONDARY :   1,701
  Class  70     SAME_LANE :   6,741
  Class  71    CROSS_ROAD :   3,616
  Class  72       DOWNGUY :   1,166



Saved → allwires_labelled_full.las
Saved → allwires_labelled_full.qml

✅  QGIS → Layer → Load Style → allwires_labelled_full.qml


## Cell 5 — Sanity check

In [5]:
print(f'Coverage per pole:\n')
print(f'{"Pole":>6}  {"Near pts":>10}  {"Labelled":>10}  {"Coverage":>10}')
print('-' * 44)
for pid in sorted(render_meta.keys()):
    mask_path = os.path.join(MASK_DIR, f'pole_{pid:03d}_mask.png')
    if not os.path.exists(mask_path): continue
    _, _, _, _, _, near_gidx = render_meta[pid]
    labelled = (new_cls[near_gidx] != cls[near_gidx]).sum()
    pct = 100.0 * labelled / max(len(near_gidx), 1)
    flag = ' ← low' if pct < 20 else ''
    print(f'{pid:>6}  {len(near_gidx):>10,}  {labelled:>10,}  {pct:>9.1f}%{flag}')
print('\nPoles with low coverage: re-open by deleting their mask file and re-running Cell 3.')


Coverage per pole:

  Pole    Near pts    Labelled    Coverage
--------------------------------------------
     1       3,975         313        7.9% ← low
     2       4,901          87        1.8% ← low
     4       4,622          87        1.9% ← low
     5       8,899         347        3.9% ← low
     6       8,778         269        3.1% ← low
     7       8,322         223        2.7% ← low
    10      10,509         621        5.9% ← low
    11      10,279         470        4.6% ← low
    12       5,939         438        7.4% ← low
    13       3,634         144        4.0% ← low
    14       3,412         172        5.0% ← low
    15       2,245         133        5.9% ← low
    16       5,292         158        3.0% ← low
    17       9,149         348        3.8% ← low
    18       8,803         348        4.0% ← low
    20      14,711       1,609       10.9% ← low
    23      13,434       1,498       11.2% ← low
    24      13,064       1,498       11.5% ← low
    25    